# Module 3: opening bio-image formats

Time: 1 hour.

There are many file formats that you might use during your career as a microscopist or bioimage analyst.
Many of these file formats contain more than just pixel data - they contain metadata including
information about the setitngs of the microscope and information about the acquisition.

Much of this information will be essential in any downstream processing!
In this section we will explore some of these formats, and the tools we can use to open them.

The BioFormats team keeps a list of file formats and some infomation about each [here](https://bio-formats.readthedocs.io/en/v8.5.0/supported-formats.html).
At current count there are more than 160 file formats, and that's just the ones that can be read by BioFormats!

### Question

How can we open different file formats?

### Objectives

- Learn about different file formats.
- Learn how to use `bioio` to extract useful metadata and load data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm
import numpy as np
import skimage

# Where we will save some example data
DATA_DIR = Path("../data")

## 1 - Format landscape

Time: 5 minutes

Some of the more common file formats you might encounter are:

- OME-TIFF: common, open, stores pixels plus OME metadata.
- OME-NGFF / OME-Zarr: chunked, multiscale, cloud/HPC-friendly image stores.
- Vendor formats: `.lif`, `.nd2`, `.czi`, `.lsm`; often need specialized
  readers and may contain multiple scenes/series.
- JPEG/PNG: useful for figures, usually poor archival formats for
  quantitative microscopy data.

When to use:

- Use vendor files when you need original acquisition metadata.
- Convert to OME-TIFF or OME-Zarr when building a reproducible analysis
  workflow that should outlive one microscope vendor.
- Use JPEG/PNG only for presentation or quick visual checks.
<div style="
  background: #fdecec;
  border-left: 6px solid #d64545;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #7f1d1d;
">
  <strong style="color: #7f1d1d;">Warning</strong><br>
 You should ALWAYS keep the original files that came off the microscope. These are primary data and should be safely stored!  </div>


## 2 - Metadata checklist

Time: 10 minutes

Before measuring anything, ask:

- What is the axis order?
- What is the dtype and bit depth?
- What are the channel names and wavelengths?
- What is the physical pixel size in x, y, and z?
- Are there multiple scenes, positions, time points, or pyramid levels?
- Was compression used? If yes, was it lossless?
- Has any pre-processing already been applied to the image? This is sometimes an option at acquisition time.
- Did the export preserve metadata or only pixel values?

## 3 - Saving and Loading images with SciKit-Image

Time: 5 minute

Sci-Kit image has an `io` module that can be used to conveniently open and save a variety of common image format with the `skimage.io.imread` and `skimage.io.imsave` functions.

Real microscopy data should usually be read with format-aware tools that preserve metadata, but the basic principle is the same: read, inspect, and only then analyze.

Nevertheless, let's save the previously introduced example data and then save it again.

In [ ]:
image = skimage.data.cells3d()

output_dir = Path("../data/outputs")
output_dir.mkdir(exist_ok=True, parents=True)

uint16_image = skimage.util.img_as_uint(image)  # make sure the image is uint16
tif_path = output_dir / "synthetic_cells.tif"
skimage.io.imsave(tif_path, uint16_image)  # save the data

loaded = skimage.io.imread(tif_path)  # load the data we just saved
print("loaded:", loaded.shape, loaded.dtype, loaded.min(), loaded.max())

<div style="
  background: #e8f7ec;
  border-left: 6px solid #2f9e44;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #1f5f2c;
">
  <strong style="color: #1f5f2c;">Question</strong><br>

  What did `imread` return?

  Is there other information related to the data which would be useful to have access to when performing analysis?
</div>

## 4 - JPEG compression is convenient, not measurement-safe

Time: 10 minutes

Lossy compression changes pixel values. That may be acceptable for a figure,
but it is usually not acceptable for intensity measurement, segmentation, or
reproducibility.

If we save the example data again, this time as a JPEG, we see that the resulting image no longer has the same pixel values as the original.

In [ ]:
image_original = loaded[30, 1, :, :]

# save the data in JPEG format
jpg_path = output_dir / "synthetic_cells.jpg"
skimage.io.imsave(jpg_path, skimage.util.img_as_ubyte(image_original), quality=25)
jpeg_loaded = skimage.io.imread(jpg_path)

# we will display the difference to see where each image is greater than the other
difference = (
    image_original.astype(float) / image_original.max()
    - jpeg_loaded.astype(float) / jpeg_loaded.max()
)
print("mean absolute JPEG difference:", np.abs(difference).mean())

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image_original, cmap="gray")
axes[0].set_title("TIFF")
axes[1].imshow(jpeg_loaded, cmap="gray")
axes[1].set_title("JPEG")
axes[2].imshow(difference, cmap="bwr", norm=CenteredNorm(), interpolation="none")
axes[2].set_title("difference")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="
  background: #fff8db;
  border-left: 6px solid #e2b200;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #8a6a00;
">
  <strong style="color: #8a6a00;">Note</strong><br>

  Where the difference display is red or blue shows where either the TIFF or JPEG image is greater, respectively.

  We see that most of the pixels did not retain there original value.
</div>

And if we look closer at the image, we can see the result of the JPEG compression. Would you want to quantify this image?

In [ ]:
fig, axes = plt.subplots(figsize=(3, 3))
axes.imshow(jpeg_loaded[150:200, 150:200], cmap="gray")

## 5 - Reader examples for real microscopy formats

Time: 10 minutes

Now we will example some vendor microscopy formats. The three you will commonly run into are `.lif` (Leica),
`.nd2` (Nikon), and `.czi` (Zeiss). Luckily, there is a python package that can read them all (and others!)

In this course we will use the `BioIO` package. The documentation can be found [here](https://bioio-devs.github.io/bioio/index.html).

We can install BioIO into our environment using .


```bash
uv add bioio
```
However, BioIO has a modular framework, so we will also need to install the individual reader plugins for each format.
We can do this all in one line by using:
```bash
uv add bioio bioio-tifffile bioio-ome-zarr bioio-lif bioio-nd2 bioio-czi
```

Once installed, the common pattern is:

```python
from bioio import BioImage

img = BioImage("path/to/file")
print(img.scenes)
print(img.dims, img.shape)
data = img.get_image_data("CZYX", T=0)
```

Here we generate a `BioImage` object that contains not just the image data, but also associated metadata.

### Let's try opening some real file formats!



In [ ]:
import logging

# Switching off some logging from bioio
logging.getLogger("bioio_lif.reader").setLevel(logging.ERROR)
logging.getLogger("bioio_czi.standard_metadata").setLevel(logging.ERROR)
logging.getLogger("bioio_nd2.reader").setLevel(logging.ERROR)

<div style="
  background: #accffb;
  border-left: 6px solid #2f80ed;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #21457f;
">
  <strong style="color: #21457f;">Exercise</strong><br>
 Use BioIO to open the supplied CZI file. Print the axis dimensions
</div>

In [ ]:
from bioio import BioImage
from python_for_ia import download_data

# Downloading an example CZI file (it might take a minute)
test_czi_path = download_data.example_czi(DATA_DIR)

# Open the czi file, inspect the dimensions and load the data.
# --- Exercise

# your code here

# ---

<div style="
  background: #e8f7ec;
  border-left: 6px solid #2f9e44;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #1f5f2c;
">
  <strong style="color: #1f5f2c;">Question</strong><br>

  Read the docs for the `BioImage.get_image_data` method.

  What happens if you request an axes that was not originally present in the data?
</div>

### Inspect the file types

<div style="
  background: #accffb;
  border-left: 6px solid #2f80ed;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #21457f;
">
  <strong style="color: #21457f;">Exercise</strong><br>

  Inspect the type of the loaded data.
</div>

In [ ]:
# --- Exercise

# your code here

# ---

<div style="
  background: #e8f7ec;
  border-left: 6px solid #2f9e44;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #1f5f2c;
">
  <strong style="color: #1f5f2c;">Question</strong><br>

  What is the type of `bioio`'s representation of an file? Why might `bioio` not want to directly return all of the image data?
</div>


### Reading Metadata

One advantage of using microscope vendor formats is that they come with a lot of metadata that can be inspected. Let's look at some of the metadata from our images.

BioIO can read the metadata from any image format it supports. You can find the documentation [here](https://bioio-devs.github.io/bioio/OVERVIEW.html#metadata-reading).

Once we have a BioImage object, we can access the metadata using

```python
metadata = img.metadata
```

In this case this returns an XML document which will have a different structure for each file format.

In [ ]:
metadata = czi.metadata

print(type(metadata))

# Manually exploring the tree
print(metadata.tag, metadata.attrib)
print(list(metadata))
print("-" * 5)
print(metadata[0].tag, metadata.attrib)
print(list(metadata[0]))
print("-" * 5)
print(metadata[0][2].tag, metadata.attrib)
print(list(metadata[0][2]))

BioIO exposes some more common useful metadata directly from the BioImage object

<div style="
  background: #accffb;
  border-left: 6px solid #2f80ed;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #21457f;
">
  <strong style="color: #21457f;">Exercise</strong><br>
 Print the X and Y pixel sizes from the metadata. Hint: Check the documentation
</div>

In [ ]:
# --- Exercise

# your code here

# ---

BioIO also exposes a list of standard metadata, which can be access using

```python
standard_metadata = czi.standard_metadata
```

From here we can see all the associated metadata, and access it diretcly using, for example:

```python
image_size_x = czi.standard_metadata.image_size_x
```


<div style="
  background: #accffb;
  border-left: 6px solid #2f80ed;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #21457f;
">
  <strong style="color: #21457f;">Exercise</strong><br>
 Print the standard metadata, and see if you can find out who collected this image
</div>

In [ ]:
# --- Exercise

# your code here

# ---

## 6 - Other file formats

Time: 10 minutes

Try opening some other file formats with BioIO. Access the metadata and see what it contains.


`.lif` example, often with multiple scenes or acquisitions:

```python
lif = BioImage("experiment.lif")
print(lif.scenes)
lif.set_scene(0)
lif_data = lif.get_image_data("CZYX", T=0)
```

`.nd2` example, often time-lapse, z-stack, or multichannel Nikon data:

```python
nd2 = BioImage("timelapse.nd2")
print(nd2.dims.order, nd2.shape)
first_time = nd2.get_image_data("CZYX", T=0)
```

<div style="
  background: #accffb;
  border-left: 6px solid #2f80ed;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #21457f;
">
  <strong style="color: #21457f;">Exercise</strong><br>

  Explore some of the example data, what can you learn from the metadata? e.g. if it has channels, can you find more information about what the channel record?

  Can you display a slice?

  <b>Tip: </b>Try to also explore the `.ome_metadata` property (and subproperties).
</div>

Use the `download_data` module that we imported above, downloading might take a minute, it contains the functions:
- `example_czi_1`
- `example_czi_2`
- `example_nd2_1`
- `example_nd2_2`
- `example_lif_1`
- `example_lif_2`

Download the data to the `DATA_DIR` path we defined above, e.g.

```python
path = download_data.example_czi_1(DATA_DIR)
```

In [ ]:
data_root = DATA_DIR
SUBDIR = "example_formats"
import pooch

public_url = "https://pub-2ac5a4b342a7472da9d44847263e1a56.r2.dev"
fname = "s_1_t_4_c_2_z_1.lif"
data_path = pooch.retrieve(
    url=public_url + f"/{fname}",
    # known_hash="97cea3e600476492e3224101d4929781f4ab4a26218c57e10b0fc1aaf5190ddba",
    fname=fname,
    path=data_root / SUBDIR,
)

In [ ]:
# --- Exercise

# your code here

# ---

<div style="
  background: #f3f4f6;
  border-left: 6px solid #6b7280;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #374151;
">
  <strong>Optional Exercise</strong><br>

  Get the OME-Zarr URL from `download_data.example_ome_zarr_url()`

  <b>Tip: </b>You can simply instantiate the `BioImage` class with the URL!
</div>

<div style="
  background: #e8f7ec;
  border-left: 6px solid #2f9e44;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
  color: #1f5f2c;
">
  <strong style="color: #1f5f2c;">Question</strong><br>

  What happens when you call `get_image_data` and you do not request all of the data?
</div>

In [ ]:
# --- Exercise

# your code here

# ---

## 7 - When a reader fails

Time: 5 minutes

Practical troubleshooting order:

1. Confirm the file opens in the vendor software or Fiji/Bio-Formats.
2. Check whether the Python reader needs a plug-in or Java/Bio-Formats.
3. Try reading a single scene or plane before loading the whole dataset.
4. Convert a copy to OME-TIFF or OME-Zarr and record the conversion command.
5. Never overwrite the raw acquisition file.